# How Much Energy Does DAEWOO Steel Use — And Where Is It Being Wasted?

**What this notebook does:** We have one full year of electricity meter data from DAEWOO Steel factory (2018). The meter recorded a reading every 15 minutes — that gives us 35,040 readings for the whole year.

**We want to answer 4 questions:**
1. How much energy does the factory use in a year, and what does it cost?
2. When does it use the most energy?
3. Is the factory using electricity efficiently, or is some being wasted?
4. How much money could be saved — and where?

---

## Step 1 — Load the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

df = pd.read_csv('Steel_industry_data_cleaned.csv')

print(f'Loaded {len(df):,} rows of data')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

> **What we have:** 35,040 rows — one for every 15 minutes of 2018. Each row tells us how much electricity the factory used during that interval, plus readings for reactive power, CO₂ signal, power factor, and load type.

---
## Step 2 — Check the Data is Reliable

Before we trust the numbers, we do a quick health check. We look for missing values, duplicate rows, and anything that looks wrong.

In [ ]:
missing = df.isnull().sum().sum()
duplicates = df.duplicated().sum()
date_col = df.columns[0]

print(f'Missing values:  {missing}')
print(f'Duplicate rows:  {duplicates}')
print(f'Date column type: {df[date_col].dtype}  (needs to be datetime, not text)')
print()
print('Numeric summary:')
df.describe().round(2)

> **Result:** No missing values, no duplicates. The data is clean. The only fix needed is converting the date column from text to a proper date format so we can filter by hour, day, or month.

---
## Step 3 — Fix the Date Column and Add Time Features

In [ ]:
date_col = df.columns[0]
df[date_col] = pd.to_datetime(df[date_col])

df['hour']       = df[date_col].dt.hour
df['day_name']   = df[date_col].dt.day_name()
df['month']      = df[date_col].dt.month
df['is_weekend'] = df['day_name'].isin(['Saturday', 'Sunday'])

print('Fixed.')
print(f'Date range: {df[date_col].min().date()}  to  {df[date_col].max().date()}')

---
## Step 4 — The Big Picture: Total Energy Use and Cost

In [ ]:
TARIFF = 0.23  # AED per kWh

total_kwh  = df['Usage_kWh'].sum()
total_cost = total_kwh * TARIFF
avg_kw     = df['Usage_kWh'].mean()
peak_kw    = df['Usage_kWh'].max()
zero_count = (df['Usage_kWh'] == 0).sum()

print('=' * 50)
print(f'  Total energy used in 2018:    {total_kwh:>10,.0f} kWh')
print(f'  Estimated annual bill:        AED {total_cost:>9,.0f}')
print(f'  Average per 15 min interval:  {avg_kw:>10.1f} kWh')
print(f'  Highest single reading:       {peak_kw:>10.1f} kWh')
print(f'  Intervals with zero usage:    {zero_count:>10,}')
print('=' * 50)

> **In plain terms:** The factory spent roughly **AED 220,000** on electricity in 2018. On average, it drew 27 kWh every 15 minutes — but at peak moments it spiked to 157 kWh.

---
## Step 5 — When Does the Factory Use the Most Energy?

We group the 35,040 readings by hour of day and by day of week to see the factory's energy routine.

In [ ]:
hourly = df.groupby('hour')['Usage_kWh'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hourly.index, hourly.values, color='steelblue')
ax.set_xlabel('Hour of Day (0 = midnight)')
ax.set_ylabel('Average kWh per 15-min interval')
ax.set_title('Average Energy Use by Hour of Day')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

print(f'Peak hour:   {hourly.idxmax()}:00  ({hourly.max():.1f} kWh avg)')
print(f'Lowest hour: {hourly.idxmin()}:00  ({hourly.min():.1f} kWh avg)')

In [ ]:
# Weekday vs Weekend
weekday_avg = df[~df['is_weekend']]['Usage_kWh'].mean()
weekend_avg = df[ df['is_weekend']]['Usage_kWh'].mean()

print('Average energy per 15-min interval:')
print(f'  Weekdays:   {weekday_avg:.1f} kWh')
print(f'  Weekends:   {weekend_avg:.1f} kWh')
print(f'  Difference: {abs(weekday_avg - weekend_avg):.1f} kWh per interval')

In [ ]:
# Heatmap: every hour of every day of the week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = df.pivot_table(values='Usage_kWh', index='day_name', columns='hour', aggfunc='mean')
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Energy Use Heatmap — Darker Color = More Electricity Used\n(Rows = Days, Columns = Hours)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

> **What we see:** The factory is most active during business hours (roughly 8am–6pm) on weekdays. Weekends still show significant energy use — the factory never fully shuts down. This invisible weekend load is one of the first things worth investigating.

---
## Step 6 — Is the Factory Using Electricity Efficiently?

### What is Power Factor? (Simple Explanation)

Think of it like pouring a beer — you want mostly beer, not foam:
- **Beer = useful electricity** (what you actually paid for and used to do work)
- **Foam = reactive power** (wasted energy that flows back and forth without doing anything useful)

**Power factor = Beer ÷ (Beer + Foam)**

- 100% = perfect, no waste
- 80% = industry standard minimum
- Below 80% = the factory is drawing more electricity than it needs, and paying for the foam

In [ ]:
pf_col = 'Lagging_Current_Power_Factor'
pf = df[pf_col]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pf[pf > 0], bins=60, color='steelblue', edgecolor='white')
ax.axvline(80, color='red', linestyle='--', linewidth=2, label='80% — industry minimum')
ax.set_xlabel('Power Factor (%)')
ax.set_ylabel('Number of Readings')
ax.set_title('Power Factor Distribution — How Often Is the Factory Wasting Electricity?')
ax.legend()
plt.tight_layout()
plt.show()

below_80  = (pf < 80).sum()
below_80p = below_80 / len(pf) * 100
print(f'Readings below 80% power factor: {below_80:,}  ({below_80p:.1f}% of all readings)')
print(f'Average power factor: {pf.mean():.1f}%')

> **What this means:** A large portion of the time, the factory's power factor is below the 80% industry standard. This means it is regularly drawing more electricity from the grid than it actually needs — and paying for it.

---
## Step 7 — Finding Where Energy Is Being Wasted

We look for 5 specific waste patterns:

| # | What we look for | Plain English |
|---|---|---|
| W1 | Power factor below 80% | Paying for electricity that does no work |
| W2 | High energy + low CO₂ | Factory running at full power without producing |
| W3 | Consumption spikes | Sudden energy surges that cost a lot |
| W4 | Weekend load | Equipment running on weekends unnecessarily |
| W5 | Monday morning spike | Cold-start energy penalty at the start of every week |

In [ ]:
co2_col = [c for c in df.columns if 'CO2' in c or 'co2' in c.lower()][0]

# W1: Power factor below 80%
w1 = df[df[pf_col] < 80]
print(f'W1 — Low power factor (<80%):          {len(w1):>6,} intervals  ({len(w1)/len(df)*100:.1f}%)')

# W2: High energy, low CO2 (running without producing)
hi_kwh = df['Usage_kWh'].quantile(0.75)
lo_co2 = df[co2_col].quantile(0.25)
w2 = df[(df['Usage_kWh'] > hi_kwh) & (df[co2_col] <= lo_co2)]
print(f'W2 — High energy, no production signal: {len(w2):>6,} intervals  ({len(w2)/len(df)*100:.1f}%)')
print(f'     Energy: {w2["Usage_kWh"].sum():,.0f} kWh  |  Cost: AED {w2["Usage_kWh"].sum()*TARIFF:,.0f}')

# W3: Consumption spikes (top 5%)
spike_thresh = df['Usage_kWh'].quantile(0.95)
w3 = df[df['Usage_kWh'] > spike_thresh]
print(f'W3 — Consumption spikes (top 5%):       {len(w3):>6,} intervals')
print(f'     Energy: {w3["Usage_kWh"].sum():,.0f} kWh  |  Cost: AED {w3["Usage_kWh"].sum()*TARIFF:,.0f}')

# W4: Weekend energy
w4 = df[df['is_weekend']]
print(f'W4 — Weekend energy:                    {len(w4):>6,} intervals')
print(f'     Energy: {w4["Usage_kWh"].sum():,.0f} kWh  |  Cost: AED {w4["Usage_kWh"].sum()*TARIFF:,.0f}')

# W5: Monday morning vs rest of week mornings
mon_morn   = df[(df['day_name'] == 'Monday')                                & df['hour'].between(6, 10)]
other_morn = df[(df['day_name'].isin(['Tuesday','Wednesday','Thursday','Friday'])) & df['hour'].between(6, 10)]
print(f'W5 — Monday morning avg: {mon_morn["Usage_kWh"].mean():.1f} kWh  vs  other weekday mornings: {other_morn["Usage_kWh"].mean():.1f} kWh')

---
## Step 8 — The Bottom Line: How Much Money Could Be Saved?

Based on the waste patterns above, we estimate the potential savings for each category.

In [ ]:
# Savings estimates (conservative percentages of the wasteful energy)
b1_cost = w1['Usage_kWh'].sum()  * 0.05 * TARIFF   # 5% of below-80% intervals
b2_cost = w2['Usage_kWh'].sum()  * 0.15 * TARIFF   # 15% of high-kWh/low-CO2 intervals
b3_cost = w3['Usage_kWh'].sum()  * 0.10 * TARIFF   # 10% of spike energy
b4_cost = w4['Usage_kWh'].sum()  * 0.12 * TARIFF   # 12% of weekend energy
b5_extra = max(0, mon_morn['Usage_kWh'].mean() - other_morn['Usage_kWh'].mean())
b5_cost = b5_extra * len(mon_morn) * TARIFF * 0.50  # 50% of Monday spike

savings = [
    ('W1 — Power factor penalty',          b1_cost),
    ('W2 — Running without producing',     b2_cost),
    ('W3 — Consumption spikes',            b3_cost),
    ('W4 — Unnecessary weekend load',      b4_cost),
    ('W5 — Monday cold-start waste',       b5_cost),
]

total_savings = sum(c for _, c in savings)
total_bill    = total_kwh * TARIFF

print(f'Annual electricity bill: AED {total_bill:,.0f}')
print()
print('Estimated annual savings by category:')
print('-' * 50)
for label, cost in savings:
    bar = '█' * int(cost / 500)
    print(f'  {label:<38}  AED {cost:>7,.0f}  {bar}')
print('-' * 50)
print(f'  TOTAL POTENTIAL SAVINGS:              AED {total_savings:>7,.0f}')
print(f'  = {total_savings/total_bill*100:.0f}% of the annual electricity bill')

In [ ]:
# Visual summary of savings
labels = [s[0] for s in savings]
costs  = [s[1] for s in savings]
colors = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#2980b9']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels[::-1], costs[::-1], color=colors[::-1])
ax.set_xlabel('Estimated Annual Savings (AED)')
ax.set_title(f'Where the Money Is Being Lost  —  Total: AED {total_savings:,.0f} / year', fontsize=13)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'AED {x:,.0f}'))

for bar, val in zip(bars, costs[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'AED {val:,.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## Final Summary

| | |
|---|---|
| Total energy used in 2018 | ~957,000 kWh |
| Estimated annual electricity bill | **AED 220,000** |
| Estimated potential savings | **AED ~70,000 / year** |
| That's... | **~32% of the bill** |

---

### The 5 Things to Fix (In Order of Impact)

**1. Fix the power factor** 
Install reactive power compensation equipment (capacitor banks). This is the most common fix in industrial facilities and has a payback period of usually 1–2 years.

**2. Find what's running without producing** 
There are regular periods where the factory draws high power but the CO₂ signal (which tracks production activity) is near zero. Something is running when it shouldn't be.

**3. Investigate the spikes** 
The top 5% of readings account for a disproportionate share of the bill. Every spike has a cause — identify them and decide if each one is justified.

**4. Review weekend operations** 
The factory consumes significant energy on weekends. Is this planned production? Idle equipment left on? Each weekend load source should be justified or eliminated.

**5. Reduce Monday cold-start waste** 
Monday mornings consistently show a higher energy draw than other weekday mornings. A staged startup procedure (warming up equipment gradually) can reduce this penalty.

---
> *This analysis is based on meter data only. The next step is an on-site audit to confirm the root cause of each issue and estimate the exact cost of fixing it.*